# OSL PPO Notebook — Bilateral Sensor + GRU backbone

Colab-ready end-to-end notebook. Edit hyperparameters directly inside each cell.

Setup
- Env: `OslEnv` — bilateral sensors (0.15 mm), independent head/body rotation, obs `[c_L, c_R, prev_v, prev_body_omega, prev_head_omega]`, action `[v, body_omega, head_omega]`.
- Actor: `GRUBackbone` (single GRU cell, hidden=421 — ≈ connectome 423-node state for scale parity) over the full 5-D obs → tanh-Gaussian. A non-connectome control for the same analysis pipeline.
- Critic: stateless 2-layer MLP over the 5-D obs.
- Trainer: custom on-policy PPO, separate actor/critic optimizers, sequence update via BPTT through the connectome.

Repo clone is mandatory — connectome CSVs live in `assets/connectome/`.

In [ ]:
# Setup — works locally and on Colab. Re-running is safe.
import os, sys, subprocess

if os.path.isdir('/content'):
    # --- Colab: clone the repo and cd in ---
    REPO_URL = 'https://github.com/InHyunseo/Brain-inspired-OSL.git'
    REPO_DIR = '/content/2d-osl'
    if not os.path.isdir(REPO_DIR):
        subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
else:
    # --- Local: find the repo root (the dir containing `src/`) by walking up ---
    REPO_DIR = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(REPO_DIR, 'src')) and REPO_DIR != os.path.dirname(REPO_DIR):
        REPO_DIR = os.path.dirname(REPO_DIR)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

for p in ('assets/connectome/weights.csv', 'assets/connectome/metadata.csv'):
    assert os.path.exists(p), f'Missing {p} — wrong cwd or clone failed?'
print('repo:', REPO_DIR, '\ncwd :', os.getcwd())

repo: /home/hyunseo/Personal_Research/OSL 
cwd : /home/hyunseo/Personal_Research/OSL


## Smoke check

In [ ]:
import torch, numpy as np
from src.envs.osl_env import OslEnv
from src.models.policy import Policy

env = OslEnv()
obs, info = env.reset(seed=0)
print('obs', obs.shape, 'action_space', env.action_space.shape)

policy = Policy(backbone='gru', gru_hidden=421)
print('backbone:', policy.backbone.describe())
print('actor params :', sum(p.numel() for p in policy.actor_parameters()))
print('critic params:', sum(p.numel() for p in policy.critic_parameters()))

obs (6,) action_space (3,)
backbone: {'backbone': 'gru', 'input_size': 6, 'hidden': 421, 'state_size': 421, 'latent_dim': 421}
actor params : 543096
critic params: 4673


## Training

All knobs are plain Python variables — change them, re-run the cell. The full plan default is `[(0,0.0,1500000),(1,0.3,500000),(1,0.6,500000),(2,1.0,1000000)]` (3.5M total). The notebook ships a smaller Colab-friendly schedule so you can iterate; bump the per-phase counts when ready for a real run.

In [4]:
# Env
ENV_KW = dict(
    sensor_spacing_mm=0.15,
    episode_seconds=120.0,
    arena_width_mm=80.0, arena_height_mm=120.0,
    source_x_mm=40.0, source_y_mm=100.0,
    gaussian_sigma_mm=30.0, success_radius_mm=7.5,
)

# Trainer (PPOConfig)
PPO_KW = dict(
    rollout_steps=128, num_envs=8, parallel_envs=True,
    update_epochs=4, minibatch_envs=4,
    gamma=0.99, gae_lambda=0.95, clip_epsilon=0.2,
    entropy_coef=0.005, value_loss_coef=0.5,
    actor_lr=3e-4, critic_lr=1e-3,
    actor_max_grad_norm=0.5, critic_max_grad_norm=0.5,
    log_std_init=-0.5,
    backbone='gru', gru_hidden=421,
    eval_interval_updates=10, eval_episodes=3,
    log_every_updates=1, checkpoint_every_timesteps=100_000,
    seed=0,
    device='auto',  # 'cpu' to force CPU
)

In [ ]:
# ===== HYPERPARAMETERS — edit freely =====
# Curriculum: list of (noise_stage, noise_strength, env_steps).
# Bump-field noise model (see src/envs/odor_field.py):
#   stage 0 = clean Gaussian, no perturbation.
#   stage 1 = STATIC bump field — many local Gaussian bumps frozen at reset.
#   stage 2 = DYNAMIC bump field — bumps drift + AR(1) amplitude oscillation
#            + occasional respawn. Hydrodynamic-feeling local turbulence.
# `strength` is the curriculum scalar α ∈ [0, 1] that linearly scales all
# bump parameters: amp_max, n_bumps, drift_speed, lifetime_inv, respawn_prob.
# Per-bump sigma is capped near the source so the global plume gradient is
# preserved even at α = 1.0.
PHASES = [
    # --- Stage 0 (clean): success-radius curriculum, spawn fixed at 55-70mm ---
    # Start with a loose goal radius (20mm) so the agent gets early successes,
    # then tighten toward the real 7.5mm target. 1M steps per radius.
    (0, 0.0, 1_500_000, 20.0),
    (0, 0.0, 500_000, 10.0),
    (0, 0.0, 500_000,  7.5),   # real target; extra steps to consolidate
    # --- Noise curriculum (goal radius fixed at 7.5mm); stop at 0.3 (beyond it fails) ---
]

# Resume: set RESUME_FROM to a checkpoint .pt to continue a stopped run; the
# curriculum loop auto-skips phases already covered by the restored step count.
# Leave '' to start a fresh run.
RESUME_FROM = ''  # e.g. 'runs/ppo_gru_nb_20260531_113633/checkpoints/ckpt_1500000.pt'
# ==========================================

import os, time, json, torch
from src.agents.ppo_agent import PPOConfig, PPOTrainer

if RESUME_FROM:
    RUN_DIR = os.path.dirname(os.path.dirname(RESUME_FROM))  # .../checkpoints/x.pt -> run dir
    print('[resume] run_dir', RUN_DIR)
else:
    RUN_DIR = os.path.join('runs', f'ppo_gru_nb_{time.strftime("%Y%m%d_%H%M%S")}')
    os.makedirs(os.path.join(RUN_DIR, 'plots'), exist_ok=True)
    with open(os.path.join(RUN_DIR, 'config.json'), 'w') as f:
        json.dump({'env': ENV_KW, 'ppo': PPO_KW, 'phases': PHASES}, f, indent=2)
    print('[run_dir]', RUN_DIR)

env_cfg = {**ENV_KW, 'noise_stage': 0, 'noise_strength': 0.0, 'seed': PPO_KW['seed']}
cfg = PPOConfig(**PPO_KW)

trainer = PPOTrainer(env_cfg, cfg, run_dir=RUN_DIR)
if RESUME_FROM:
    trainer.load_checkpoint(RESUME_FROM)
summary = {}
try:
    for stage, strength, steps, goal_radius in PHASES:
        print(f'\n=== Phase noise_stage={stage} strength={strength} goal_radius={goal_radius} steps={steps} ===')
        trainer.runner.set_noise_stage(stage, strength)
        trainer.runner.set_success_radius(goal_radius)
        trainer.env_config['success_radius_mm'] = goal_radius  # so eval uses the same goal
        summary = trainer.train(phase_timesteps=steps)
    trainer.save_final(summary)
    print('\n[final summary]\n' + json.dumps(summary, indent=2))
finally:
    trainer.close()


## Training curves

In [ ]:
# ===== Training curve — presentation figure (Slide 4) =====
# Smoothed success ratio vs steps with phase 0/1/2 markers.
# Saves to RUN_DIR/analysis/slide4_training_curves.png and shows it inline.
import json, os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image as DisplayImage, display

plt.rcParams.update({
    "font.size": 10, "axes.titlesize": 12, "axes.labelsize": 10,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9,
})
FIGSIZE = (6, 4.5)
FIG_DIR = os.path.join(RUN_DIR, "analysis")
os.makedirs(FIG_DIR, exist_ok=True)

rows = [json.loads(l) for l in open(os.path.join(RUN_DIR, "training_log.jsonl"))]
ev = [(r["total_steps"], r["eval_success_rate"]) for r in rows if "eval_success_rate" in r]
ev_x = [s / 1e6 for s, _ in ev]
ev_y = [y for _, y in ev]

def smooth(y, w=151):
    y = np.asarray(y, dtype=float)
    if len(y) < 3:
        return y
    w = min(w, len(y) if len(y) % 2 == 1 else len(y) - 1)
    if w < 3:
        return y
    pad = w // 2
    return np.convolve(np.pad(y, pad, mode="edge"), np.ones(w) / w, mode="valid")

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(ev_x, smooth(ev_y), color="#1f77b4", lw=2.4)
ax.set_xlabel("Steps (M)")
ax.set_ylim(-0.02, 1.02)
ax.set_title("Success ratio")
ax.grid(alpha=0.3)
# phase boundaries (edit if your curriculum changes)
for b in (1.5, 2.0):
    ax.axvline(b, color="gray", ls="--", lw=1.2)
for cx, name in ((0.75, "phase 0"), (1.75, "phase 1"), (2.25, "phase 2")):
    ax.text(cx, 1.04, name, ha="center", va="bottom", fontsize=10, color="dimgray")
fig.tight_layout()
out = os.path.join(FIG_DIR, "slide4_training_curves.png")
fig.savefig(out, dpi=150)
plt.close(fig)
print("saved", out)
display(DisplayImage(data=open(out, "rb").read(), format="png"))


## Evaluation + Elite-seed GIF
1. `find_elite_seeds(...)` — 시드를 순회하면서 (a) 성공 종료, (b) episode 내 cast 이벤트(`event_is_high_cast_like`) 횟수가 `[min_casts, max_casts]` 범위인 시드만 모아 `N`개 확보.
2. `render_elite_gif(seed=...)` — 원하는 elite 시드 하나를 골라(`elites[i]` 또는 직접 정수) GIF로 렌더링.

`min_casts` / `max_casts`로 너무 헤매는 / 너무 직진하는 시드를 걸러낼 수 있어요.


In [ ]:
# ===== EVAL HYPERPARAMETERS =====
EVAL_NOISE_STAGE = 2
EVAL_NOISE_STRENGTH = 0.3      # bump-field α at eval time
EVAL_SEED_BASE = 20_000
EVAL_MAX_TRIALS = 500          # upper bound on seeds to scan when finding elites
N_ELITES = 5                   # collect this many elite seeds
ELITE_MIN_CASTS = 1            # episode must contain at least this many cast events
ELITE_MAX_CASTS = 300          # ... and at most this many (filters out pure spinning)
ELITE_PICK_INDEX = 0           # which elite to render (0 = first found). Set to None
                               # to override with EXPLICIT_SEED below.
EXPLICIT_SEED = None           # if not None, skip elite search and render this seed directly
# ================================


import os, glob, torch, numpy as np
from IPython.display import Image as DisplayImage, display

from src.envs.osl_env import EnvConfig, OslEnv
from src.agents.ppo_agent import PPOConfig
from src.models.policy import Policy
from src.utils.plotter import render_rollout_frame, save_gif

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Pick the latest checkpoint in this run (robust to whatever final step count training reached).
_ckpts = sorted(glob.glob(os.path.join(RUN_DIR, 'checkpoints', '*.pt')))
assert _ckpts, f'no checkpoints under {RUN_DIR}/checkpoints'
ckpt_path = _ckpts[-1]
print('using checkpoint', ckpt_path)
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
cfg = PPOConfig(**ckpt['agent_config'])
policy = Policy(
    weights_csv=cfg.weights_csv, metadata_csv=cfg.metadata_csv,
    latent_dim=cfg.latent_dim, message_passing_steps=cfg.message_passing_steps,
    log_std_init=cfg.log_std_init,
    backbone=cfg.backbone, gru_hidden=cfg.gru_hidden,
).to(device)
policy.load_state_dict(ckpt['policy_state_dict']); policy.eval()

def make_eval_env(seed):
    cfg_dict = {**ENV_KW, 'noise_stage': EVAL_NOISE_STAGE,
                'noise_strength': EVAL_NOISE_STRENGTH, 'seed': seed}
    return OslEnv(EnvConfig.from_dict(cfg_dict))

def _run_episode(seed, collect_traj=False):
    """Roll out one deterministic episode; return summary (+ optional trajectory)."""
    env = make_eval_env(seed)
    obs, _ = env.reset(seed=seed)
    actor_state, critic_state = policy.initial_states(1, device)
    mask = torch.zeros(1, 1, device=device)
    ret, casts, success = 0.0, 0, False
    traj_x, traj_y, cast_x, cast_y, frames = [], [], [], [], []
    for t in range(env.max_steps):
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            a, _, _, n_as, n_cs = policy.act(obs_t, actor_state, critic_state, mask, deterministic=True)
        if collect_traj:
            traj_x.append(env.x_mm); traj_y.append(env.y_mm)
        obs, r, term, trunc, info = env.step(a.squeeze(0).cpu().numpy())
        ret += float(r)
        if info.get('event_is_high_cast_like'):
            casts += 1
            if collect_traj:
                cast_x.append(env.x_mm); cast_y.append(env.y_mm)
        if collect_traj:
            frames.append(render_rollout_frame(env, traj_x, traj_y, cast_x, cast_y, t,
                                               title=f'PPO seed={seed} step={t} casts={casts}'))
        done = bool(term or trunc)
        if done:
            success = bool(info.get('success', False))
            mask.fill_(0.0)
        else:
            mask.fill_(1.0)
        actor_state = n_as * mask; critic_state = n_cs * mask
        if done:
            break
    return {
        'seed': seed, 'return': ret, 'success': success, 'casts': casts,
        'frames': frames if collect_traj else None,
    }

def find_elite_seeds(n_to_find=N_ELITES, min_casts=ELITE_MIN_CASTS,
                     max_casts=ELITE_MAX_CASTS, max_trials=EVAL_MAX_TRIALS,
                     seed_base=EVAL_SEED_BASE):
    print(f'🔍 elite seed 탐색 (success + casts ∈ [{min_casts}, {max_casts}], '
          f'α={EVAL_NOISE_STRENGTH}, up to {max_trials} trials)…')
    elites, trial = [], 0
    while len(elites) < n_to_find and trial < max_trials:
        seed = seed_base + trial
        r = _run_episode(seed, collect_traj=False)
        if r['success'] and min_casts <= r['casts'] <= max_casts:
            elites.append(r)
            print(f"  ✨ seed={seed} return={r['return']:.2f} casts={r['casts']}")
        trial += 1
    print(f'→ {len(elites)} elite seed(s) after {trial} trials')
    return elites

# ---- Elite search OR explicit seed ----
if EXPLICIT_SEED is not None:
    chosen_seed = int(EXPLICIT_SEED)
    print(f'using EXPLICIT_SEED={chosen_seed}')
else:
    elites = find_elite_seeds()
    if not elites:
        raise RuntimeError('No elite seed met the criteria — relax min_casts/max_casts '
                           'or raise EVAL_MAX_TRIALS.')
    pick = ELITE_PICK_INDEX
    pick = min(pick, len(elites) - 1)
    chosen_seed = elites[pick]['seed']
    print(f'\nelite seeds: {[e["seed"] for e in elites]}')
    print(f'rendering elites[{pick}] = seed {chosen_seed}')

# ---- Render the chosen seed ----
result = _run_episode(chosen_seed, collect_traj=True)
print(f"seed={result['seed']} return={result['return']:.2f} success={result['success']} casts={result['casts']}")
gif_path = os.path.join(RUN_DIR, 'plots', f'agent_seed{chosen_seed}.gif')
os.makedirs(os.path.dirname(gif_path), exist_ok=True)
save_gif(result['frames'], gif_path, fps=15)
display(DisplayImage(data=open(gif_path, 'rb').read(), format='gif'))


## Mechanistic analysis (connectome / GRU interpretability)

학습된 정책의 **신경회로 수준 기전**을 분석합니다 (`Analysis.osl2d` 파이프라인 — 3D `osl_analysis`에서 이식). `RUN_DIR/ckpt_final.pt`를 deterministic 롤아웃해 hidden-state trace를 덤프한 뒤 4-phase 분석을 실행합니다:

- **Phase 1** 행동 라벨(run/cast/turn/spin/stop) 분포 + 전이행렬 + cast PSD
- **Phase 2a** hidden state PCA→UMAP 분리도 (silhouette / Calinski-Harabasz)
- **Phase 2b** episode-split linear probe (행동이 hidden에 선형 디코딩되는가)
- **Phase 2c** per-neuron 기여도 → 라벨별 top-K 뉴런 (`neuron_groups.json`)
- **Phase 3a** Jacobian eigenmode (slow attractor / oscillatory mode)
- **Phase 3b** fixed-point 탐색 + 안정성
- **Phase 4** neuron-group ablation 인과 검증 (라이브 env)

> Colab에서 UMAP 시각화를 쓰려면 `pip install umap-learn` (없으면 자동으로 PCA-2 fallback).

In [ ]:
# ===== Analysis Cell 1: dump traces + run the full 4-phase pipeline =====
# Re-uses RUN_DIR from training. Analyzes ckpt_final.pt (label "final").
# Tune ANALYSIS_* knobs for speed vs. coverage.


ANALYSIS_SEEDS = [0, 1]
ANALYSIS_EPISODES_PER_SEED = 8
ANALYSIS_MAX_STEPS = None          # None = full episodes
ANALYSIS_NOISE_STAGE = 2
ANALYSIS_NOISE_STRENGTH = 0.2
ANALYSIS_STOCHASTIC = True         # sample actions: deterministic mean stalls on weak odor signal
ANALYSIS_TOP_K = 16                # top-K neurons per label for Phase 2c/4
ANALYSIS_SAMPLES_PER_LABEL = 60    # Jacobian samples per label (Phase 3a)
RUN_ABLATION = True                # Phase 4 (live env) — set False to skip
ABLATION_EPS = 3
# =======================================================================

import os
from Analysis.osl2d import run_all as _run_all

_argv = [
    '--run-dir', RUN_DIR,
    '--checkpoints', 'final',
    '--seeds', *[str(s) for s in ANALYSIS_SEEDS],
    '--episodes-per-seed', str(ANALYSIS_EPISODES_PER_SEED),
    '--noise-stage', str(ANALYSIS_NOISE_STAGE),
    '--noise-strength', str(ANALYSIS_NOISE_STRENGTH),
    '--top-k', str(ANALYSIS_TOP_K),
    '--samples-per-label', str(ANALYSIS_SAMPLES_PER_LABEL),
    '--ablation-eps', str(ABLATION_EPS),
]
if ANALYSIS_STOCHASTIC:
    _argv += ['--stochastic']
if ANALYSIS_MAX_STEPS is not None:
    _argv += ['--max-steps', str(ANALYSIS_MAX_STEPS)]
if not RUN_ABLATION:
    _argv += ['--skip-ablation']

_run_all.main(_argv)
print('\n[analysis] artifacts →', os.path.join(RUN_DIR, 'analysis'))

[run_all] collecting traces for ckpt='final' (stochastic) ...
[dump] ckpt=final__stoch seed=0 ep=000 T= 949 return=   20.77 success=1
[dump] ckpt=final__stoch seed=0 ep=001 T=1200 return=   -0.62 success=0
[dump] ckpt=final__stoch seed=0 ep=002 T= 845 return=   19.94 success=1
[dump] ckpt=final__stoch seed=0 ep=003 T= 803 return=   19.54 success=1
[dump] ckpt=final__stoch seed=0 ep=004 T=1189 return=   -4.50 success=0
[dump] ckpt=final__stoch seed=0 ep=005 T= 683 return=   20.10 success=1
[dump] ckpt=final__stoch seed=0 ep=006 T= 913 return=   20.62 success=1
[dump] ckpt=final__stoch seed=0 ep=007 T= 497 return=   20.78 success=1
[dump] ckpt=final__stoch seed=1 ep=000 T= 466 return=   20.62 success=1
[dump] ckpt=final__stoch seed=1 ep=001 T= 441 return=   20.55 success=1
[dump] ckpt=final__stoch seed=1 ep=002 T= 994 return=   19.93 success=1
[dump] ckpt=final__stoch seed=1 ep=003 T=1032 return=   19.04 success=1
[dump] ckpt=final__stoch seed=1 ep=004 T= 814 return=   19.08 success=1
[d

/home/hyunseo/anaconda3/envs/osl/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[run_all] phase2b ...
[run_all] phase2c ...
[run_all] phase3a ...
[run_all] phase3b ...
[run_all] phase4 ablation (ckpt=final) ...
[run_all] DONE → runs/ppo_gru_nb_20260531_113633/analysis/

[analysis] artifacts → runs/ppo_gru_nb_20260531_113633/analysis


In [ ]:
# ===== Analysis Cell 2: presentation figures only =====
# Builds ONLY the slide figures into RUN_DIR/analysis/ and shows them inline:
#   slide6_active_sensing_overlap.png   — active-sensing top-k overlap vs clean
#   slide6_jacobian_run.png / _active_sensing.png — eigenvalue clouds (dominant mode in red)
#   slide6_jacobian_oscillation.png     — dominant-mode oscillation fraction (Run vs AS)
#   slide2_baseline_noise_sweep.png     — baseline success vs noise (only if a baseline run exists)
# Reuses the ANALYSIS_* knobs and cached traces from the cell above. No other figures.
import os, json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image as DisplayImage, display, Markdown

plt.rcParams.update({
    "font.size": 10, "axes.titlesize": 12, "axes.labelsize": 10,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9,
})
FIGSIZE = (6, 4.5)
BLUE, RED = "#1f77b4", "#d62728"
FIG_DIR = os.path.join(RUN_DIR, "analysis")
os.makedirs(FIG_DIR, exist_ok=True)
shown = []

from pathlib import Path
from Analysis.osl2d._io import adapter_for_ckpt, load_traces
from Analysis.osl2d.jacobian import jacobian_at
from Analysis.osl2d.segment import LABEL_TO_INT

# ---------------------------------------------------------------- (1) overlap vs noise
# Active-sensing top-k neuron set, how much it overlaps the clean (0.0) set as noise grows.
import Analysis.noise_sweep_cast as nsc
nsc.RUN_DIR = Path(RUN_DIR)
nsc.OUT = Path(FIG_DIR)
nsc.CKPT_LABEL = "final"
nsc.NOISE_STAGE = ANALYSIS_NOISE_STAGE
nsc.NOISE_STRENGTHS = [0.0, 0.1, 0.2, 0.3]
nsc.SEEDS = tuple(ANALYSIS_SEEDS)
nsc.EPISODES_PER_SEED = ANALYSIS_EPISODES_PER_SEED
nsc.MAX_STEPS = ANALYSIS_MAX_STEPS if ANALYSIS_MAX_STEPS is not None else 1200
nsc.TOP_K = ANALYSIS_TOP_K
nsc.DEVICE = "cpu"

xs = nsc.NOISE_STRENGTHS
topk = {a: nsc.collect_and_topk(a) for a in xs}
clean = topk[xs[0]]["top"].get("ACTIVE_SENSING", topk[xs[0]]["top"].get("CAST", []))
overlap = [nsc.jaccard(clean, topk[a]["top"].get("ACTIVE_SENSING",
                       topk[a]["top"].get("CAST", []))) for a in xs]

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(xs, overlap, "-o", color=BLUE, lw=2.4, ms=8)
ax.set_xlabel(f"Noise (phase {ANALYSIS_NOISE_STAGE})")
ax.set_ylim(-0.02, 1.04)
ax.set_title("Neuron overlap")
ax.grid(alpha=0.3)
fig.tight_layout()
p = os.path.join(FIG_DIR, "slide6_active_sensing_overlap.png")
fig.savefig(p, dpi=150); plt.close(fig); shown.append(p)
print("overlap vs clean:", [round(v, 2) for v in overlap])

# ---------------------------------------------------------------- (2) Jacobian: clean-field eigenvalues
# Dump/load a clean (0.0) stochastic trace set; per timestep take the dominant
# eigenvalue (largest |lambda|) of the hidden Jacobian. Off the real axis => oscillation.
clean_label = topk[0.0]["label"]            # 'final_n00__stoch' (already dumped above)
traces = load_traces(RUN_DIR, [clean_label])
adapter = adapter_for_ckpt(RUN_DIR, clean_label, device="cpu")
rng = np.random.default_rng(0)
SAMPLES = ANALYSIS_SAMPLES_PER_LABEL

def eigs_for(name):
    lab = LABEL_TO_INT[name]
    idx = np.where(traces.label == lab)[0]
    if len(idx) == 0:
        return np.array([]), np.array([])
    pick = rng.choice(idx, size=min(SAMPLES, len(idx)), replace=False)
    allw, dom = [], []
    for t in pick:
        w = np.linalg.eigvals(jacobian_at(adapter, traces.obs[t], traces.h[t]))
        allw.append(w); dom.append(w[np.argmax(np.abs(w))])
    return np.concatenate(allw), np.asarray(dom)

theta = np.linspace(0, 2 * np.pi, 360)
osc_frac = {}
for key, title, fn in [("RUN", "Run", "slide6_jacobian_run.png"),
                       ("ACTIVE_SENSING", "Active sensing", "slide6_jacobian_active_sensing.png")]:
    E_all, E_dom = eigs_for(key)
    osc_frac[title] = float(np.mean(np.abs(E_dom.imag) > 1e-6)) if len(E_dom) else 0.0
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    if len(E_all):
        ax.scatter(E_all.real, E_all.imag, s=6, alpha=0.25, color="0.7", label="all modes")
        ax.scatter(E_dom.real, E_dom.imag, s=22, alpha=0.8, color=RED,
                   edgecolors="none", label="dominant mode")
    ax.plot(np.cos(theta), np.sin(theta), color="black", lw=0.6, alpha=0.4)
    ax.axhline(0, color="k", lw=0.3, alpha=0.4); ax.axvline(0, color="k", lw=0.3, alpha=0.4)
    ax.set_xlim(-1.15, 1.15); ax.set_ylim(-1.15, 1.15); ax.set_aspect("equal")
    ax.set_title(title); ax.set_xlabel("Re"); ax.set_ylabel("Im")
    ax.legend(loc="upper left", framealpha=0.9, markerscale=1.3)
    fig.tight_layout()
    p = os.path.join(FIG_DIR, fn); fig.savefig(p, dpi=150); plt.close(fig); shown.append(p)
print("oscillation fraction:", {k: round(v, 2) for k, v in osc_frac.items()})

# ---------------------------------------------------------------- (3) oscillation: two points + line
fig, ax = plt.subplots(figsize=FIGSIZE)
ys = [osc_frac.get("Run", 0.0), osc_frac.get("Active sensing", 0.0)]
ax.plot([0, 1], ys, "-", color="gray", lw=2, zorder=1)
ax.scatter([0], [ys[0]], s=140, color=BLUE, zorder=2)
ax.scatter([1], [ys[1]], s=140, color=RED, zorder=2)
for x, y in zip([0, 1], ys):
    ax.text(x, y + 0.04, f"{y:.2f}", ha="center", fontsize=11)
ax.set_xticks([0, 1]); ax.set_xticklabels(["Run", "Active sensing"])
ax.set_xlim(-0.4, 1.4); ax.set_ylim(0, max(0.65, max(ys) + 0.12))
ax.set_title("Oscillation"); ax.grid(axis="y", alpha=0.3)
ax.text(0.5, max(ys) + 0.08, "Active sensing oscillates; Run just decays",
        ha="center", fontsize=10, color="dimgray")
fig.tight_layout()
p = os.path.join(FIG_DIR, "slide6_jacobian_oscillation.png")
fig.savefig(p, dpi=150); plt.close(fig); shown.append(p)

# ---------------------------------------------------------------- (4) baseline (optional)
# If a hand-built chemotaxis baseline run exists, draw its success-vs-noise curve.
BASE_DIR = os.path.join("runs", "baseline_chemotaxis")
base_json = os.path.join(BASE_DIR, "noise_sweep.json")
if os.path.exists(base_json):
    ns = json.load(open(base_json))
    want = [(0, 0.0), (2, 0.3), (2, 0.6), (2, 1.0)]
    sel = []
    for st, sg in want:
        for r in ns:
            if r["stage"] == st and abs(r["strength"] - sg) < 1e-6:
                sel.append(r); break
    if sel:
        fig, ax = plt.subplots(figsize=FIGSIZE)
        ax.plot(range(len(sel)), [r["success_rate"] for r in sel], "-o", color=BLUE, lw=2.4, ms=8)
        ax.set_xticks(range(len(sel)))
        ax.set_xticklabels([f'{r["strength"]:.1f}' for r in sel])
        ax.set_xlabel("Noise (phase 2)"); ax.set_ylim(0, 1.04)
        ax.set_title("Success ratio"); ax.grid(alpha=0.3)
        fig.tight_layout()
        p = os.path.join(FIG_DIR, "slide2_baseline_noise_sweep.png")
        fig.savefig(p, dpi=150); plt.close(fig); shown.append(p)
else:
    print("[baseline] no runs/baseline_chemotaxis/noise_sweep.json — skipping Slide-2 figure")

# ---------------------------------------------------------------- show
display(Markdown("### Presentation figures (saved to `RUN_DIR/analysis/`)"))
for p in shown:
    display(Markdown(f"**{os.path.basename(p)}**"))
    display(DisplayImage(data=open(p, "rb").read(), format="png"))
